In [119]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "laumer2018spontaneous")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Laumer_ 2018_bending_task.csv")
complete_path_2 = os.path.join(original_data_pathway, "Laumer2018_unbending_task_Spontaneous hook.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [120]:
import pandas as pd
import numpy as np
import pyreadstat

df1 = pd.read_csv(complete_path_1)
df2 = pd.read_csv(complete_path_2)

experiment_import = [[df1, 'bending_task', 'hook_bending_task'],
                    [df2, 'unbending_task', 'unbending_task']]
for x,y,k in experiment_import:
    x["condition"]=y
    x["successful_trials"]=k
# df1["experiment"]="1"
# df2["experiment"]="2"
# df2.columns


df1.rename(columns={"Latency between Start and first toch wire (seconds)": "Latency between Start and first touch wire (seconds)"}, inplace=True)
# df1.columns

In [121]:
import re
replace_1=re.compile('( |\;|-|\:|\/|\(|\))') # ' ', -, ( and )
replacement_list = [[df1, '(yes=1, no=0)', ''],
                [df1, '(yes=1,no=0)', ''],
                [df1, '(1=yes, 0=no)',''],
                [df1, '(seconds)', '_in_seconds'],
                [df1, '(1=at apparatus; 2=elsewhere)', ''],
                [df1, replace_1, '_'],
                [df1, '__', '_'],
                [df1, '__', '_'],
                [df2, '(yes=1, no=0)', ''],
                [df2, '(yes=1,no=0)', ''],
                [df2, '(1=yes, 0=no)',''],
                [df2, '(seconds)', '_in_seconds'],
                [df2, '(1=at apparatus; 2=elsewhere)', ''],
                [df2, replace_1, '_'],
                [df2, '__', '_'],
                [df2, '__', '_']]
for x, y, k in replacement_list:
    x.columns = x.columns.str.replace(y, k, regex=True)


In [122]:
strip_list = [df1, df2]
for x in strip_list:
    x.columns = x.columns.str.rstrip('_')


In [123]:
data_frames=[df1, df2]
for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x.columns = x.columns.str.replace(replace_1, ' ')
    x=x.rename(columns={"subject": "ape"})
    x['study_id']="laumer2018spontaneous"
    x=x.rename(columns={'session ':'session'})
    data_frames[index]=x
new_df=data_frames[0]
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)
# fulldf.columns

In [124]:
code_list=["preexperience_in_the_hook_bending_test", 
    "preexperience_with_raking_tools_during_previous_studies", 
    "condition_basket_fixed", 
    "tool_turned_around",
    "string_touched", 
    "wire_insterted_in_wrong_tube", 
    "pre_experience_received"]

for index, x in enumerate(code_list):    
    fulldf[x] = fulldf[x].astype(str)
    temp=[]
    for entry in fulldf[x]:
        if entry == '0' or entry =='0.0':
            entry = "no"
        elif entry =='1' or entry =='1.0':
            entry = "yes"
        temp.append(entry)
    fulldf = fulldf.assign(temp_col=temp)
    fulldf=fulldf.rename(columns={'temp_col': x+'_codes'})

In [125]:
fulldf = fulldf.rename(columns={"place of modification (1=at apparatus; 2=elsewhere)": "place_of_modification"})
code_list=["place_of_modification"]
temp=[]
for index, x in enumerate(code_list):    
    fulldf[x] = fulldf[x].astype(str)
    for entry in fulldf[x]:
        if entry == '1' or entry =='1.0':
            entry = "at_apparatus"
        elif entry =='2' or entry=='2.0':
            entry = "elsewhere"
        else:
            entry = entry
        temp.append(entry)
    fulldf = fulldf.assign(temp_col=temp)
fulldf=fulldf.rename(columns={'temp_col': x+'_codes'})



In [126]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
fulldf['ape'] = fulldf['ape'].str.rstrip()
# for x,y in zip(df_name['wrong'],df_name['right']):
#     fulldf['ape'].replace(x, y, inplace=True)
fulldf.replace('nan', np.nan, inplace=True)

In [127]:
comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
fulldf= fulldf.merge(apedf,left_on='ape', right_on='name', how='left')
# fulldf.columns
fulldf.rename(columns={"ape": "participant"}, inplace=True)

In [128]:
order_of_task = [['padana', 'bending'], ['tanah', 'bending'], ['pini', 'bending'],
                ['dokana', 'unbending'], ['raja', 'unbending'], ['bimbo', 'unbending']]
fulldf['first_task']=""
for x, y in order_of_task:
    fulldf.loc[fulldf.participant == x, ['first_task']] = y

In [129]:

fulldf = fulldf.rename(columns={'tool_crafting_time_excluding_interruptions_in_in_seconds':'tool_crafting_time_excluding_interruptions_in_seconds',
    'time_until_success_excluding_time_when_subject_was_not_interacting_with_material_apparatus_in_in_seconds':'time_until_success_excluding_time_when_subject_was_not_interacting_with_material_apparatus_in_seconds'})

replace_list_2= [['pre_experience', 'preexperience'],
                ['insterted','inserted'],
                ['inserstion','insertion']]
for x,y in replace_list_2:
    fulldf.columns = fulldf.columns.str.replace(x, y, regex=True)


In [130]:
fulldf['technique_used'].replace(',', '_', inplace=True, regex=True)

In [131]:
spe_2=[]  
for index, row in fulldf.iterrows():
    if not pd.isna(row['duration_probing_with_unmodified_wire_in_seconds']):
        spe_2.append(row['duration_probing_with_unmodified_wire_in_seconds'])
    else:
        spe_2.append(row['duration_probing_with_unmodified_tool'])
fulldf = fulldf.assign(duration_probing_with_unmodified_tool_in_seconds=spe_2)

In [132]:
spe_3=[]  
for index, row in fulldf.iterrows():
    if not pd.isna(row['preexperience_in_the_hook_bending_test_codes']):
        spe_3.append(row['preexperience_in_the_hook_bending_test_codes'])
    else:
        spe_3.append(row['preexperience_received_codes'])
fulldf = fulldf.assign(received_preexperience_during_this_study=spe_3)

In [133]:
modification_list_2= [['1.2', 'at_apparatus_then_elsewhere'],
                ['2.1','elsewhere_then_at_apparatus'],
                ['0.0',np.nan]]
for x,y in modification_list_2:
    fulldf['place_of_modification_codes'].replace(x, y, inplace=True, regex=True)

# fulldf['place_of_modification_codes'].unique()

In [134]:
fulldf=fulldf[['study_id',  'participant', 'sex','species','session', 'trial',
        'condition','first_task',
        # 'successful_trials',
#        'preexperience_in_the_hook_bending_test_codes',
        'received_preexperience_during_this_study',
       'preexperience_with_raking_tools_during_previous_studies_codes',
#        'preexperience_received_codes',  
       'condition_basket_fixed_codes',
       'time_until_success_excluding_time_when_subject_was_not_interacting_with_material_apparatus_in_seconds',
#        'duration_probing_with_unmodified_wire_in_seconds','duration_probing_with_unmodified_tool',
       'duration_probing_with_unmodified_tool_in_seconds',
       'duration_of_probing_with_functional_modified_tool_in_seconds',
       'duration_of_probing_with_modified_non_functional_tool_in_seconds',
       'latency_between_start_and_first_modification_wire_in_seconds',
       'tool_crafting_time_excluding_interruptions_in_seconds',
       'place_of_modification_codes', 'technique_used', 'tool_turned_around_codes',
       'how_often_tool_turned_around', 'string_touched_codes',
       'duration_of_string_manipulation_in_seconds',
       'wire_inserted_in_wrong_tube_codes',
       'duration_of_wrong_insertion_in_seconds', 'string_inserted',
       'duration_of_string_insertion_in_seconds', 
        
       'latency_between_start_and_first_touch_wire_in_seconds']]

fulldf.columns =fulldf.columns.str.replace('_codes', '')

In [135]:
fulldf.at[29, 'technique_used'] = 'r'

In [136]:
comp_out_path = os.path.join(out_pathway, 'laumer2018spontaneous_standardized.csv')
fulldf.to_csv(comp_out_path, encoding='utf-8-sig', index=False)

names = fulldf.columns.tolist()
exp_g = pd.DataFrame(names)
exp_g = exp_g.rename(columns={0: "column_name"})
exp_g["description"] = ""
exp_g=exp_g[["column_name", "description"]]
comp_out_path_glossary = os.path.join(out_pathway, 'laumer2018spontaneous_glossary.csv')
exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)

# for index in range(1,3):
#     exp = fulldf[fulldf['experiment'] == str(index)]
#     exp = exp.dropna(axis=1, how='all')
#     comp_out_path = os.path.join(out_pathway, 'laumer2018spontaneous_exp'+str(index)+'_standardized.csv')
#     exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
#     names = exp.columns.tolist()
#     exp_g = pd.DataFrame(names)
#     exp_g = exp_g.rename(columns={0: "column_name"})
#     exp_g["description"] = ""
#     exp_g=exp_g[["column_name", "description"]]
#     comp_out_path_glossary = os.path.join(out_pathway, 'laumer2018spontaneous_exp'+str(index)+'_glossary.csv')
#     exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)